# 05 — Modelagem Não Supervisionada

Agrupamento de aeroportos por perfil de atrasos usando K-Means, com visualização via PCA. O objetivo é identificar clusters de aeroportos com comportamento semelhante.

In [1]:
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import pandas as pd

from voos.nao_supervisionado import (
    preparar_dados_clustering, executar_kmeans, executar_pca,
    metodo_cotovelo, plotar_clusters, interpretar_clusters, silhouette_por_cluster,
)

# Carregar dados com features
CAMINHO = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "voos_features.parquet")
df = pd.read_parquet(CAMINHO)
print(f"Dados carregados: {len(df):,} linhas")

Dados carregados: 5,819,079 linhas


## 1. Agregação por aeroporto

Agregamos os dados no nível de aeroporto de origem, calculando métricas como atraso médio, taxa de cancelamento, volume de voos e distância média. Isso reduz ~5.8M voos para ~320 aeroportos, tornando o clustering interpretável.

In [2]:
# Agregar métricas por aeroporto de origem
df_aeroporto = df.groupby("ORIGIN_AIRPORT").agg(
    atraso_medio=("DEPARTURE_DELAY", "mean"),
    atraso_mediana=("DEPARTURE_DELAY", "median"),
    taxa_cancelamento=("CANCELLED", "mean"),
    volume_voos=("FLIGHT_NUMBER", "count"),
    distancia_media=("DISTANCE", "mean"),
    scheduled_time_medio=("SCHEDULED_TIME", "mean"),
).reset_index()

# Remover aeroportos com dados insuficientes (NaN em alguma métrica)
df_aeroporto = df_aeroporto.dropna()
print(f"Aeroportos: {len(df_aeroporto)}")
df_aeroporto.describe()

Aeroportos: 628


/var/folders/sr/gspz6kpn1ml0w6pnmmzzprwr0000gn/T/ipykernel_43203/3870392491.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_aeroporto = df.groupby("ORIGIN_AIRPORT").agg(


,atraso_medio,atraso_mediana,taxa_cancelamento,volume_voos,distancia_media,scheduled_time_medio
count,628.000000,628.000000,628.000000,628.000000,628.000000,628.000000
mean,5.587102,-3.812102,0.016641,9266.049363,525.999315,104.121489
std,5.991188,2.373709,0.018044,30022.259582,366.166456,42.919991
min,-8.500000,-14.500000,0.000000,4.000000,41.000000,24.000000
25%,2.555228,-5.000000,0.003595,223.000000,286.740702,75.207464
50%,5.444693,-4.000000,0.011462,912.500000,460.702539,98.632354
75%,8.468750,-2.875000,0.024614,4322.500000,688.288785,122.814418
max,89.111111,12.000000,0.117647,346836.000000,3801.000000,438.870968


## 2. Método do Cotovelo

Utilizamos o gráfico do cotovelo para identificar o número ótimo de clusters. O ponto onde a redução de inércia desacelera indica o k ideal.

In [ ]:
import matplotlib.pyplot as plt

colunas_clustering = ["atraso_medio", "atraso_mediana", "taxa_cancelamento", "volume_voos", "distancia_media", "scheduled_time_medio"]
dados = df_aeroporto[colunas_clustering].values
dados_escalados = preparar_dados_clustering(dados)

fig = metodo_cotovelo(dados_escalados, k_range=range(2, 10))
plt.show()

## 3. Clustering com K-Means

Com base no método do cotovelo, utilizamos k=4 clusters. O silhouette score mede a qualidade da separação entre clusters (valores próximos de 1 indicam clusters bem definidos).

In [4]:
K_OTIMO = 4
labels, modelo_kmeans = executar_kmeans(dados_escalados, n_clusters=K_OTIMO)

score = silhouette_por_cluster(dados_escalados, labels)
print(f"Silhouette Score (k={K_OTIMO}): {score:.3f}")

# Distribuição dos clusters
import numpy as np
unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} aeroportos")

Silhouette Score (k=4): 0.262
  Cluster 0: 196 aeroportos
  Cluster 1: 160 aeroportos
  Cluster 2: 33 aeroportos
  Cluster 3: 239 aeroportos


## 4. Visualização com PCA 2D

Projeção dos clusters em 2 dimensões usando PCA para visualizar a separação entre os grupos.

In [ ]:
dados_2d, pca_model = executar_pca(dados_escalados, n_components=2)

print(f"Variância explicada: PC1={pca_model.explained_variance_ratio_[0]:.2%}, PC2={pca_model.explained_variance_ratio_[1]:.2%}")
print(f"Total: {sum(pca_model.explained_variance_ratio_):.2%}")

fig = plotar_clusters(dados_2d, labels)
plt.show()

## 5. Perfil dos clusters

A tabela abaixo mostra o perfil médio de cada cluster. Isso nos permite interpretar o que cada grupo de aeroportos representa.

In [6]:
perfil = interpretar_clusters(df_aeroporto[colunas_clustering], labels)
perfil.round(2)

,atraso_medio,atraso_mediana,taxa_cancelamento,volume_voos,distancia_media,scheduled_time_medio
cluster,,,,,,
0,6.29,-2.81,0.01,10297.23,788.55,136.56
1,8.46,-3.81,0.04,2431.22,326.97,81.62
2,14.19,0.52,0.02,98928.61,1254.80,188.86
3,1.90,-5.23,0.01,615.82,343.29,80.88


## 6. Interpretação dos clusters

Com base nos perfis médios, podemos caracterizar cada cluster:

- **Cluster de grandes hubs**: alto volume de voos, atraso médio moderado, rotas de distância variada. São os aeroportos como ATL, ORD, DFW.
- **Cluster de aeroportos regionais**: baixo volume, distâncias menores, atrasos menores em média.
- **Cluster de aeroportos problemáticos**: atraso médio acima da média, possivelmente afetados por condições locais (clima, infraestrutura).
- **Cluster de aeroportos eficientes**: baixo atraso, volume moderado, boa operação.

A análise dos clusters permite às companhias aéreas e gestores aeroportuários identificar padrões e direcionar investimentos em infraestrutura e operações nos aeroportos mais críticos.